> **Optional.** The server already works with the pretrained YOLOv8m pothole model that `setup.sh` downloads (`scripts/download_pretrained.py`). Use this notebook later to train on your own roads. Tip: set `BASE_MODEL = '../models/pothole.pt'` to fine-tune that pothole model instead of starting from plain YOLO26m.

> Saving here **replaces** `models/pothole.pt`; keep a copy of the pretrained file if you want to compare.

# Train YOLO26m to find potholes

Fine-tunes the pretrained **YOLO26m** on pothole data and installs the result into the server's `models/` folder.

**Where to run:** a CUDA GPU (Google Colab, Kaggle) trains about 10–20× faster than the Mac. Training on the M1 works (`mps`) but expect hours. If you train elsewhere, copy `best.pt` back to `models/pothole.pt` and run the last section on the Mac to export CoreML.

**Data:** put each source dataset in its own folder under `data/training/raw/`:

```
data/training/raw/
  hrp4k/     HRP4K (Hugging Face: dronefreak/HRP4K)        YOLO or COCO format
  rdd2022/   RDD2022 road damage, keep class D40 (potholes)  Pascal VOC (xml)
  own/       frames from your own phone, labelled            YOLO format
```

Check each dataset's licence before commercial use. A few hundred labelled frames from your own phone and roads usually helps more than anything else.

## Setup

In [ ]:
# On Colab / a fresh machine, uncomment:
# !pip install -q "ultralytics>=8.4.0" huggingface_hub matplotlib

import hashlib, json, os, platform, random, shutil, sys
import xml.etree.ElementTree as ET
from pathlib import Path

import torch
from ultralytics import YOLO, __version__ as ul_version

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DEVICE = 0 if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('ultralytics', ul_version, '| torch', torch.__version__, '| device', DEVICE)

## Settings

In [ ]:
BASE_MODEL = 'yolo26m.pt'   # downloaded automatically on first use
EPOCHS     = 100
IMGSZ      = 640
BATCH      = 16 if DEVICE == 0 else 8   # lower it if you run out of memory
PATIENCE   = 25            # stop early if validation stops improving
SEED       = 42
SPLIT      = (0.8, 0.1, 0.1)   # train / val / test
BG_KEEP    = 0.15          # share of pothole-free images kept as negatives

WORK = ROOT / 'data' / 'training'
RAW  = WORK / 'raw'
OUT  = WORK / 'pothole'      # merged dataset in YOLO format
RAW.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
print('raw data in', RAW)

## Optional: download HRP4K from Hugging Face

Set `DOWNLOAD_HRP4K = True` to fetch it. Then look at the folder tree it prints and pick the matching converter in the next section (YOLO `.txt` labels, or a COCO `.json`).

In [ ]:
DOWNLOAD_HRP4K = False

def tree(path, depth=2, prefix=''):
    path = Path(path)
    if depth < 0 or not path.exists(): return
    items = sorted(path.iterdir())
    dirs = [p for p in items if p.is_dir()]
    files = [p for p in items if p.is_file()]
    for d in dirs[:12]:
        print(f'{prefix}{d.name}/  ({sum(1 for _ in d.rglob("*") if _.is_file())} files)')
        tree(d, depth - 1, prefix + '    ')
    for f in files[:5]:
        print(f'{prefix}{f.name}')

if DOWNLOAD_HRP4K:
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id='dronefreak/HRP4K', repo_type='dataset', local_dir=RAW / 'hrp4k')
tree(RAW)

## Converters → one class: `pothole` (id 0)

Each converter collects `(image path, YOLO label lines)` pairs. Images with no pothole are kept as background with probability `BG_KEEP`, so the model also learns what is *not* a pothole (shadows, patches, manholes).

In [ ]:
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
samples = []   # dicts: src, image, lines

def _keep(lines):
    return bool(lines) or random.random() < BG_KEEP

def add_yolo(folder, pothole_ids=(0,), name=None):
    """YOLO layout: .../images/x.jpg with .../labels/x.txt (class cx cy w h, normalised)."""
    folder, n0 = Path(folder), len(samples)
    for img in folder.rglob('*'):
        if img.suffix.lower() not in IMG_EXT: continue
        parts = list(img.parts)
        if 'images' not in parts: continue
        i = len(parts) - 1 - parts[::-1].index('images')
        lbl = Path(*parts[:i], 'labels', *parts[i + 1:]).with_suffix('.txt')
        lines = []
        if lbl.exists():
            for row in lbl.read_text().split('\n'):
                v = row.split()
                if len(v) >= 5 and int(float(v[0])) in pothole_ids:
                    lines.append('0 ' + ' '.join(v[1:5]))
        if _keep(lines):
            samples.append({'src': name or folder.name, 'image': img, 'lines': lines})
    print(f'{name or folder.name}: {len(samples) - n0} images')

def add_voc(folder, keep_names=('D40',), name=None):
    """Pascal VOC: one .xml per image (RDD2022: D00/D10/D20 cracks, D40 pothole)."""
    folder, n0 = Path(folder), len(samples)
    images = {p.stem: p for p in folder.rglob('*') if p.suffix.lower() in IMG_EXT}
    for xml in folder.rglob('*.xml'):
        root = ET.parse(xml).getroot()
        img = images.get(Path(root.findtext('filename', xml.stem)).stem) or images.get(xml.stem)
        if img is None: continue
        W = float(root.findtext('size/width') or 0); H = float(root.findtext('size/height') or 0)
        if not W or not H: continue
        lines = []
        for obj in root.iter('object'):
            if obj.findtext('name', '').strip() not in keep_names: continue
            b = obj.find('bndbox')
            x1, y1, x2, y2 = (float(b.findtext(k)) for k in ('xmin', 'ymin', 'xmax', 'ymax'))
            lines.append(f'0 {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} {(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}')
        if _keep(lines):
            samples.append({'src': name or folder.name, 'image': img, 'lines': lines})
    print(f'{name or folder.name}: {len(samples) - n0} images')

def add_coco(json_path, image_dir, keep_names=('pothole',), name=None):
    """COCO detection json (bbox = x, y, w, h in pixels)."""
    data, n0 = json.loads(Path(json_path).read_text()), len(samples)
    cats = {c['id'] for c in data['categories'] if c['name'].lower() in {k.lower() for k in keep_names}}
    by_img = {}
    for a in data['annotations']:
        if a['category_id'] in cats: by_img.setdefault(a['image_id'], []).append(a['bbox'])
    for im in data['images']:
        img = Path(image_dir) / im['file_name']
        if not img.exists(): continue
        W, H = im['width'], im['height']
        lines = [f'0 {(x + w / 2) / W:.6f} {(y + h / 2) / H:.6f} {w / W:.6f} {h / H:.6f}'
                 for x, y, w, h in by_img.get(im['id'], [])]
        if _keep(lines):
            samples.append({'src': name or Path(json_path).stem, 'image': img, 'lines': lines})
    print(f'{name or Path(json_path).stem}: {len(samples) - n0} images')

## Collect the sources you have

Edit this cell to match what you downloaded. Missing folders are skipped. For a YOLO dataset with several classes, pass the class id(s) that mean pothole (check its `data.yaml`).

In [ ]:
samples.clear()
if (RAW / 'hrp4k').exists():
    coco = sorted((RAW / 'hrp4k').rglob('*.json'))
    if any((RAW / 'hrp4k').rglob('labels')):
        add_yolo(RAW / 'hrp4k', pothole_ids=(0,), name='hrp4k')
    elif coco:
        for j in coco:   # one json per split is common
            add_coco(j, j.parent / 'images' if (j.parent / 'images').exists() else j.parent, name=f'hrp4k-{j.stem}')
    else:
        print('hrp4k: unknown layout, look at the tree above and call a converter by hand')
if (RAW / 'rdd2022').exists():
    add_voc(RAW / 'rdd2022', keep_names=('D40',), name='rdd2022')
if (RAW / 'own').exists():
    add_yolo(RAW / 'own', pothole_ids=(0,), name='own')

with_pothole = sum(1 for s in samples if s['lines'])
print(f'\nTotal {len(samples)} images, {with_pothole} with potholes, '
      f'{sum(len(s["lines"]) for s in samples)} boxes')
assert with_pothole > 50, 'Too few pothole images: add data under data/training/raw/'

## Split and write the merged dataset

In [ ]:
def split_of(path):
    # stable split: the same image always lands in the same split
    h = int(hashlib.md5(str(path).encode()).hexdigest(), 16) % 1000 / 1000
    return 'train' if h < SPLIT[0] else 'val' if h < SPLIT[0] + SPLIT[1] else 'test'

if OUT.exists(): shutil.rmtree(OUT)
counts = {}
for i, s in enumerate(samples):
    sp = split_of(s['image'])
    stem = f"{s['src']}_{i:06d}"
    (OUT / 'images' / sp).mkdir(parents=True, exist_ok=True)
    (OUT / 'labels' / sp).mkdir(parents=True, exist_ok=True)
    dst = OUT / 'images' / sp / (stem + s['image'].suffix.lower())
    try: os.link(s['image'], dst)          # hard link: no extra disk space
    except OSError: shutil.copy2(s['image'], dst)
    (OUT / 'labels' / sp / (stem + '.txt')).write_text('\n'.join(s['lines']))
    counts[(sp, s['src'])] = counts.get((sp, s['src']), 0) + 1

DATA_YAML = OUT / 'data.yaml'
DATA_YAML.write_text(f'''path: {OUT}
train: images/train
val: images/val
test: images/test
names:
  0: pothole
''')
import pandas as pd
pd.Series(counts).unstack(0).fillna(0).astype(int)

## Look at a few labelled images

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

picks = random.sample([s for s in samples if s['lines']], k=min(6, with_pothole))
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, s in zip(axes.flat, picks):
    im = Image.open(s['image']).convert('RGB'); W, H = im.size; d = ImageDraw.Draw(im)
    for ln in s['lines']:
        _, cx, cy, w, h = map(float, ln.split())
        d.rectangle([(cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H], outline=(255, 122, 26), width=4)
    ax.imshow(im); ax.set_title(s['src']); ax.axis('off')
plt.tight_layout()

## Train

In [ ]:
model = YOLO(BASE_MODEL)
results = model.train(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE,
    patience=PATIENCE, seed=SEED, project=str(WORK / 'runs'), name='yolo26m_pothole', exist_ok=True,
    cos_lr=True, close_mosaic=10,
    # road-friendly augmentation: lighting changes yes, upside-down roads no
    hsv_h=0.015, hsv_s=0.6, hsv_v=0.5, fliplr=0.5, flipud=0.0, degrees=0.0, scale=0.5,
)
BEST = Path(results.save_dir) / 'weights' / 'best.pt'
print('best weights:', BEST)

## Test on held-out images

In [ ]:
best = YOLO(str(BEST))
m = best.val(data=str(DATA_YAML), split='test', imgsz=IMGSZ, device=DEVICE)
print(f'test  mAP50 {m.box.map50:.3f} | mAP50-95 {m.box.map:.3f} | precision {m.box.mp:.3f} | recall {m.box.mr:.3f}')

## Try it on your own footage

Point `VIDEO` at a dashcam/phone clip. `vid_stride=6` checks every 6th frame (≈5 fps from a 30 fps video), like the scanner. The annotated video is saved under `runs/`.

In [ ]:
VIDEO = None   # e.g. ROOT / 'data' / 'test_drive.mp4'
if VIDEO:
    best.predict(source=str(VIDEO), conf=0.4, imgsz=IMGSZ, vid_stride=6, save=True,
                 project=str(WORK / 'runs'), name='video_check', exist_ok=True, device=DEVICE)

## Install into the server

Copies `best.pt` to `models/pothole.pt`. On a Mac it also exports CoreML (`.mlpackage`), which the server prefers because it runs on the Neural Engine. Restart the server afterwards.

In [ ]:
MODELS = ROOT / 'models'; MODELS.mkdir(exist_ok=True)
shutil.copy2(BEST, MODELS / 'pothole.pt')
print('copied ->', MODELS / 'pothole.pt')

if platform.system() == 'Darwin':
    exported = Path(YOLO(str(MODELS / 'pothole.pt')).export(format='coreml', imgsz=IMGSZ, half=True))
    target = MODELS / 'pothole.mlpackage'
    if target.exists(): shutil.rmtree(target)
    shutil.move(str(exported), target)
    print('CoreML ->', target)
else:
    print('Not on a Mac: copy models/pothole.pt to the Mac and run  python scripts/export_coreml.py')

## If results are weak

- **Low recall (misses potholes):** add more of *your own* frames, especially far-away and wet potholes; try `IMGSZ = 800`.
- **Low precision (false alarms):** add background frames with shadows, patches, manholes and speed breakers to `own/` with empty label files.
- **Good test score, bad on the road:** the public data doesn't look like your camera. Label 300–500 frames from the scanner itself (the server saves them in `data/photos/`).